# Fire Season Timing | Global

In [33]:
'''
Computes fire season timing metrics (onset, peak, end, season length) for all WWF RESOLVE
ecoregions globally for years 2003-2025. Exports daily fire counts to Google Drive as
one CSV per ecoregion per year. Post-run assembly and metric computation happen after
manual download of Drive files.

Data sources:
- MODIS Terra active fire: MODIS/061/MOD14A1
- MODIS Aqua active fire:  MODIS/061/MYD14A1
- Ecoregions:              RESOLVE/ECOREGIONS/2017

Region definition:
- Global — all WWF RESOLVE ecoregions (~847 after removing Rock and Ice)
- Subsetting: use TEST_N / TEST_IDS to run on a reduced set for testing

Output (all paths derived from RUN_LABEL, RUN_VERSION, and BASE_OUT_DIR):
- <BASE_OUT_DIR>/runs/<RUN_LABEL>_<RUN_VERSION>/fire_metrics/daily_counts/  ← downloaded CSVs go here
- <BASE_OUT_DIR>/runs/<RUN_LABEL>_<RUN_VERSION>/fire_metrics/_all_daily_counts.csv
- <BASE_OUT_DIR>/runs/<RUN_LABEL>_<RUN_VERSION>/fire_metrics/_all_metrics.csv
- <BASE_OUT_DIR>/runs/<RUN_LABEL>_<RUN_VERSION>/fire_metrics/_eco_quality.csv
- <BASE_OUT_DIR>/runs/<RUN_LABEL>_<RUN_VERSION>/fire_metrics/master_<RUN_LABEL>_<RUN_VERSION>.csv
- <BASE_OUT_DIR>/runs/<RUN_LABEL>_<RUN_VERSION>/README.txt
- <BASE_OUT_DIR>/runs/<RUN_LABEL>_<RUN_VERSION>/eco_geometries.json
'''

import ee
import pandas as pd
import numpy as np
import os
import time
import datetime
import calendar
import json
import glob
from tqdm import tqdm

In [34]:
# Authenticate and initialize ----------------------------------------------------------------------
ee.Authenticate()
ee.Initialize(project='fire-seasons')

In [35]:
# RUN CONFIGURATION --------------------------------------------------------------------------------
# Set these before running anything else. All output paths are derived from these values.

RUN_LABEL   = 'global'  # short name for this run
RUN_VERSION = 'v1'         # increment this for each new run
RUN_NOTES   = """
Testing the global pipeline with 10 ecoregions
"""

In [36]:
# Folder structure and paths -----------------------------------------------------------------------

BASE_OUT_DIR = r'C:\Users\ibekar\Documents\GitProjects\TGPF'  # Windows
# BASE_OUT_DIR = '/Users/ibekar/Github/TGPF'                  # Mac

_run_stamp = datetime.date.today().strftime('%Y-%m-%d')
_run_name  = f'{RUN_LABEL}_{RUN_VERSION}'
run_dir    = os.path.join(BASE_OUT_DIR, 'runs', _run_name)
raw_dir    = os.path.join(run_dir, 'raw')
output_dir = os.path.join(run_dir, 'fire_metrics')
daily_dir  = os.path.join(output_dir, 'daily_counts')

os.makedirs(output_dir, exist_ok=True)

print(f'Run name  : {_run_name}')
print(f'Run dir   : {run_dir}')
print(f'Raw dir   : {raw_dir}')
print(f'Output dir: {output_dir}')

Run name  : global_v1
Run dir   : C:\Users\ibekar\Documents\GitProjects\TGPF\runs\global_v1
Raw dir   : C:\Users\ibekar\Documents\GitProjects\TGPF\runs\global_v1\raw
Output dir: C:\Users\ibekar\Documents\GitProjects\TGPF\runs\global_v1\fire_metrics


## Setup

In [37]:
# LOAD MODIS COLLECTIONS ---------------------------------------------------------------------------
# Terra and Aqua are loaded once here at module level.
# Per-year and per-day filtering is handled inside get_daily_counts().

terra = ee.ImageCollection("MODIS/061/MOD14A1").select('FireMask')
aqua  = ee.ImageCollection("MODIS/061/MYD14A1").select('FireMask')

print('Terra image count:', terra.size().getInfo())
print('Aqua image count:', aqua.size().getInfo())
print('Terra and Aqua collections loaded.')

Terra image count: 9471
Aqua image count: 8651
Terra and Aqua collections loaded.


In [38]:
# LOAD GLOBAL ECOREGIONS ---------------------------------------------------------------------------
ecoregions = ee.FeatureCollection("RESOLVE/ECOREGIONS/2017")

n_eco      = ecoregions.size().getInfo()
batch_size = 100
offset     = 0
eco_list   = []

print(f'Total ecoregions: {n_eco}. Fetching in batches of {batch_size}...')

while offset < n_eco:
    batch = (ecoregions
             .select(['ECO_ID', 'ECO_NAME', 'BIOME_NUM', 'BIOME_NAME'])
             .toList(batch_size, offset)
             .getInfo())
    eco_list.extend(batch)
    offset += batch_size
    print(f'  Fetched {len(eco_list)} / {n_eco}')

print(f'Done. {len(eco_list)} ecoregion features loaded.')

Total ecoregions: 848. Fetching in batches of 100...
  Fetched 100 / 848
  Fetched 200 / 848
  Fetched 300 / 848
  Fetched 400 / 848
  Fetched 500 / 848
  Fetched 600 / 848
  Fetched 700 / 848
  Fetched 800 / 848
  Fetched 848 / 848
Done. 848 ecoregion features loaded.


In [39]:
# BUILD GLOBAL ECO RECORDS -------------------------------------------------------------------------
eco_records = []
removed = []
for f in eco_list:
    p = f['properties']
    if p['ECO_ID'] == 0:
        removed.append(p)
        continue
    eco_records.append({
        'eco_id'    : p['ECO_ID'],
        'eco_name'  : p['ECO_NAME'],
        'biome_num' : p['BIOME_NUM'],
        'biome_name': p['BIOME_NAME'],
        'geometry'  : ee.Geometry(f['geometry'])
    })

print(f'Built {len(eco_records)} ecoregion records.')
print(f'Removed {len(removed)} ecoregion(s):')
for r in removed:
    print(f"  {r['ECO_ID']} | {r['ECO_NAME']} | {r['BIOME_NAME']}")

Built 846 ecoregion records.
Removed 2 ecoregion(s):
  0 | Rock and Ice | N/A
  0 | Rock and Ice | N/A


## Parameters

In [42]:
# PARAMETERS ---------------------------------------------------------------------------------------

FIRE_MASK_MIN   = 8     # FireMask threshold: >= 8 = nominal + high confidence only
ONSET_THRESHOLD = 0.05  # Cumulative fraction threshold for fire season onset (5%)
END_THRESHOLD   = 0.95  # Cumulative fraction threshold for fire season end (95%)
MIN_DETECTIONS  = 20    # Minimum annual fire detections required to compute metrics
YEARS           = list(range(2003, 2026))  # Full study period: 2003–2025

# BIMODALITY DIAGNOSTICS ---------------------------------------------------------------------------
# Applied only when season_length > MIN_SEASON_FOR_BIMODALITY days.
# Soft flag: one metric triggers. Hard flag: both trigger.
MIN_SEASON_FOR_BIMODALITY = 90    # Minimum season length (days) before bimodality is assessed
BC_THRESHOLD              = 0.555 # Bimodality coefficient above this → bimodality signal

In [51]:
# SUBSETTING (set to None to disable) --------------------------------------------------------------
TEST_N   = 5
TEST_IDS = None

# APPLY SUBSETTING ---------------------------------------------------------------------------------
eco_run = eco_records

if TEST_IDS is not None:
    eco_run = [e for e in eco_run if e['eco_id'] in TEST_IDS]
    print(f'Subsetting to {len(eco_run)} ecoregions by ID: {TEST_IDS}')

if TEST_N is not None:
    eco_run = eco_run[:TEST_N]
    print(f'Subsetting to first {TEST_N} ecoregions.')

print(f'Running pipeline on {len(eco_run)} / {len(eco_records)} ecoregions.')

Subsetting to first 5 ecoregions.
Running pipeline on 5 / 846 ecoregions.


In [52]:
# # SAVE GEOMETRIES TO DISK --------------------------------------------------------------------------
# # Saves ecoregion geometries as GeoJSON for reuse in visualization notebooks
# # without needing a GEE connection. Skipped if file already exists.
# # Uses eco_run and respects subsetting if active, full list if not.

# os.makedirs(run_dir, exist_ok=True)
# geo_path = os.path.join(run_dir, 'eco_geometries.json')

# if os.path.exists(geo_path):
#     print(f'Geometries already saved. Skipping. ({geo_path})')
# else:
#     geo_records_export = []
#     for rec in eco_run:
#         geo_records_export.append({
#             'eco_id'  : rec['eco_id'],
#             'eco_name': rec['eco_name'],
#             'geometry': rec['geometry'].getInfo() # simplfy unit is meters
#         })

#     with open(geo_path, 'w') as f:
#         json.dump(geo_records_export, f)

#     print(f'Saved {len(geo_records_export)} geometries → {geo_path}')

In [44]:
# WRITE README -------------------------------------------------------------------------------------
_readme_path = os.path.join(run_dir, 'README.txt')
with open(_readme_path, 'w') as _f:
    _f.write(f'Run name    : {_run_name}\n')
    _f.write(f'Date        : {_run_stamp}\n')
    _f.write(f'Years       : {YEARS[0]}–{YEARS[-1]}\n')
    _f.write(f'TEST_N      : {TEST_N}\n')
    _f.write(f'TEST_IDS    : {TEST_IDS}\n')
    _f.write(f'Ecoregions  : {len(eco_run)} / {len(eco_records)}\n')
    _f.write(f'\nNotes:\n{RUN_NOTES.strip()}\n')
print(f'README written → {_readme_path}')

README written → C:\Users\ibekar\Documents\GitProjects\TGPF\runs\global_v1\README.txt


## Helper Functions

In [53]:
# HELPER FUNCTIONS ---------------------------------------------------------------------------------

# Constant empty fallback image — defined once, shared by both functions
_empty = ee.Image.constant(0).rename('FireMask').toUint8()


def build_fire_fc_year(eco, year):
    """
    Builds a server-side GEE FeatureCollection of daily fire detection counts
    for one ecoregion for ONE year. Used for complex geometries (>50k vertices)
    and as fallback for single-task failures.
    """
    eco_id   = eco['eco_id']
    eco_name = eco['eco_name']
    geometry = eco['geometry'].simplify(500)

    start   = ee.Date.fromYMD(year, 1, 1)
    end     = ee.Date.fromYMD(year + 1, 1, 1)
    n_days  = 366 if calendar.isleap(year) else 365

    terra_year = terra.filterDate(start, end)
    aqua_year  = aqua.filterDate(start, end)
    day_seq    = ee.List.sequence(0, n_days - 1)

    def make_daily_feature(d):
        d        = ee.Number(d)
        date     = start.advance(d, 'day')
        date_end = date.advance(1, 'day')

        t = ee.Image(ee.Algorithms.If(
            terra_year.filterDate(date, date_end).size().gt(0),
            terra_year.filterDate(date, date_end).select('FireMask').max(),
            _empty
        ))
        a = ee.Image(ee.Algorithms.If(
            aqua_year.filterDate(date, date_end).size().gt(0),
            aqua_year.filterDate(date, date_end).select('FireMask').max(),
            _empty
        ))

        count = t.max(a).gte(FIRE_MASK_MIN).unmask(0).reduceRegion(
            reducer   = ee.Reducer.sum(),
            geometry  = geometry,
            scale     = 1000,
            maxPixels = 1e9,
            bestEffort= True
        ).get('FireMask')

        return ee.Feature(None, {
            'eco_id'      : eco_id,
            'eco_name'    : eco_name,
            'year'        : year,
            'doy'         : d.add(1).toInt(),
            'n_detections': ee.Number(count).toInt()
        })

    return ee.FeatureCollection(day_seq.map(make_daily_feature))


def build_fire_fc(eco):
    """
    Builds a server-side GEE FeatureCollection of daily fire detection counts
    for one ecoregion across ALL years. Used for simple geometries (≤50k vertices).
    Merges per-year FeatureCollections into one flat collection.
    """
    eco_id   = eco['eco_id']
    eco_name = eco['eco_name']
    geometry = eco['geometry'].simplify(500)

    years_list = ee.List(YEARS)

    def process_year(year):
        year    = ee.Number(year).toInt()
        start   = ee.Date.fromYMD(year, 1, 1)
        end     = ee.Date.fromYMD(year.add(1), 1, 1)
        n_days  = end.difference(start, 'day').toInt()

        terra_year = terra.filterDate(start, end)
        aqua_year  = aqua.filterDate(start, end)
        day_seq    = ee.List.sequence(0, n_days.subtract(1))

        def make_daily_feature(d):
            d        = ee.Number(d)
            date     = start.advance(d, 'day')
            date_end = date.advance(1, 'day')

            t = ee.Image(ee.Algorithms.If(
                terra_year.filterDate(date, date_end).size().gt(0),
                terra_year.filterDate(date, date_end).select('FireMask').max(),
                _empty
            ))
            a = ee.Image(ee.Algorithms.If(
                aqua_year.filterDate(date, date_end).size().gt(0),
                aqua_year.filterDate(date, date_end).select('FireMask').max(),
                _empty
            ))

            count = t.max(a).gte(FIRE_MASK_MIN).unmask(0).reduceRegion(
                reducer   = ee.Reducer.sum(),
                geometry  = geometry,
                scale     = 1000,
                maxPixels = 1e9,
                bestEffort= True
            ).get('FireMask')

            return ee.Feature(None, {
                'eco_id'      : eco_id,
                'eco_name'    : eco_name,
                'year'        : year,
                'doy'         : d.add(1).toInt(),
                'n_detections': ee.Number(count).toInt()
            })

        return ee.FeatureCollection(day_seq.map(make_daily_feature))

    return ee.FeatureCollection(years_list.map(process_year)).flatten()

In [54]:
# PRE-CLASSIFY ECOREGIONS BY GEOMETRY COMPLEXITY --------------------------------------------------
# Uses geometries already in eco_list (fetched earlier) — no GEE calls needed.
# VERTEX_THRESHOLD is empirical — tune after first run.

VERTEX_THRESHOLD = 50000

def count_vertices_from_geojson(geojson):
    '''Count total coordinate points from a GeoJSON geometry dict.'''
    geom_type = geojson.get('type', '')
    coords    = geojson.get('coordinates', [])

    if geom_type == 'Polygon':
        return sum(len(ring) for ring in coords)
    elif geom_type == 'MultiPolygon':
        return sum(len(ring) for poly in coords for ring in poly)
    return 0

# Build vertex lookup from eco_list
vertex_lookup = {}
for f in eco_list:
    p      = f['properties']
    eco_id = p['ECO_ID']
    if eco_id == 0:
        continue
    vertex_lookup[eco_id] = count_vertices_from_geojson(f['geometry'])

# Attach to eco_run
for eco in eco_run:
    eco['n_vertices'] = vertex_lookup.get(eco['eco_id'], 0)
    eco['split_task'] = eco['n_vertices'] > VERTEX_THRESHOLD

n_single_pre = sum(1 for e in eco_run if not e['split_task'])
n_split_pre  = sum(1 for e in eco_run if e['split_task'])

print(f'Ecoregions pre-classified:')
print(f'  Single task (≤{VERTEX_THRESHOLD} vertices): {n_single_pre}')
print(f'  Split/per-year (>{VERTEX_THRESHOLD} vertices): {n_split_pre}')
print()
print('Top 10 most complex geometries:')
sorted_ecos = sorted(eco_run, key=lambda e: e['n_vertices'], reverse=True)
for e in sorted_ecos[:10]:
    print(f"  {e['eco_id']:4d} | {e['n_vertices']:6d} vertices | {e['eco_name']}")

Ecoregions pre-classified:
  Single task (≤50000 vertices): 4
  Split/per-year (>50000 vertices): 1

Top 10 most complex geometries:
   112 |  71921 vertices | East African mangroves
   322 |   9085 vertices | Sunda Shelf mangroves
   321 |   4783 vertices | Myanmar Coast mangroves
   111 |   2617 vertices | Central African mangroves
   616 |   2490 vertices | Southern Atlantic Brazilian mangroves


## Main Pipeline

In [55]:
# SUBMIT FIRE EXPORT TASKS -------------------------------------------------------------------------
# Pre-classified ecoregions by vertex count:
#   - Simple (≤50k vertices) → one task covering all years
#   - Complex (>50k vertices) → one task per year
#
# If a single task fails at submission despite being pre-classified as simple,
# it is logged and skipped — does NOT fall back to per-year automatically.
# Check _submitted_tasks.txt for FAILED entries after submission completes.
#
# Naming convention:
#   Single task : FIRE_<RUN_ID>_<run_name>_eco_<eco_id>_allyears
#   Per-year    : FIRE_<RUN_ID>_<run_name>_eco_<eco_id>_yr_<year>
#
# When complete: move all CSVs from Drive/<DRIVE_FOLDER>/ to:
#   <output_dir>/daily_counts/
#
# Monitor at: https://code.earthengine.google.com/tasks

DRIVE_FOLDER = 'fire_daily_counts'
RUN_ID       = datetime.date.today().strftime('%Y%m%d')

os.makedirs(output_dir, exist_ok=True)
os.makedirs(daily_dir,  exist_ok=True)

submitted_log = os.path.join(output_dir, '_submitted_tasks.txt')

# Reset log at the start of each submission run
with open(submitted_log, 'w') as log:
    log.write(f'Submission started: {datetime.datetime.now().strftime("%Y-%m-%d %H:%M")}\n')
    log.write(f'Run: {_run_name} | RUN_ID: {RUN_ID}\n')
    log.write(f'{"-" * 60}\n')

n_submitted = 0
n_single    = 0
n_split     = 0

for eco in tqdm(eco_run, desc='Submitting'):
    eco_id    = eco['eco_id']
    eco_name  = eco['eco_name']
    safe_name = eco_name.replace(' ', '_').replace('/', '_')

    if not eco['split_task']:
        # --- Single task: all years ---
        try:
            task_desc = f'FIRE_{RUN_ID}_{_run_name}_eco_{eco_id}_allyears'
            file_name = f'{eco_id}_{safe_name}_allyears_daily'

            daily_fc = build_fire_fc(eco)

            task = ee.batch.Export.table.toDrive(
                collection    = daily_fc,
                description   = task_desc,
                folder        = DRIVE_FOLDER,
                fileNamePrefix= file_name,
                fileFormat    = 'CSV',
                selectors     = ['eco_id', 'eco_name', 'year', 'doy', 'n_detections']
            )
            task.start()
            n_submitted += 1
            n_single    += 1
            print(f'  [{n_submitted}] {eco_name} ({eco_id}): \
                  single task (n_vertices={eco["n_vertices"]:,})')


            with open(submitted_log, 'a') as log:
                log.write(f'{eco_id} | all | {task_desc}\n')

        except Exception as e:
            print(f'  WARNING: {eco_name} ({eco_id}) single task failed — {e}')
            with open(submitted_log, 'a') as log:
                log.write(f'{eco_id} | FAILED_SINGLE | {str(e)}\n')
            continue

    else:
        # --- Per-year tasks: complex geometry ---
        for year in YEARS:
            try:
                task_desc = f'FIRE_{RUN_ID}_{_run_name}_eco_{eco_id}_yr_{year}'
                file_name = f'{eco_id}_{safe_name}_{year}_daily'

                daily_fc = build_fire_fc_year(eco, year)

                task = ee.batch.Export.table.toDrive(
                    collection    = daily_fc,
                    description   = task_desc,
                    folder        = DRIVE_FOLDER,
                    fileNamePrefix= file_name,
                    fileFormat    = 'CSV',
                    selectors     = ['eco_id', 'eco_name', 'year', 'doy', 'n_detections']
                )
                task.start()
                n_submitted += 1
                print(f'  [{n_submitted}] {eco_name} ({eco_id}): \
                       year {year} (n_vertices={eco["n_vertices"]:,})')


                with open(submitted_log, 'a') as log:
                    log.write(f'{eco_id} | {year} | {task_desc}\n')

            except Exception as e2:
                print(f'    {eco_id} | {year}: FAILED — {e2}')
                with open(submitted_log, 'a') as log:
                    log.write(f'{eco_id} | {year} | FAILED: {e2}\n')

        n_split += 1
        print(f'  → {eco_name} ({eco_id}): all {len(YEARS)} years submitted (split)')

print(f'\nSubmission complete.')
print(f'  Total tasks submitted : {n_submitted}')
print(f'  Single-task ecoregions: {n_single}')
print(f'  Split ecoregions      : {n_split}')
print(f'\nCheck {submitted_log} for any FAILED entries.')
print(f'Monitor at: https://code.earthengine.google.com/tasks')
print(f'\nWhen complete, move CSVs from Drive/{DRIVE_FOLDER}/ to:')
print(f'  {daily_dir}')

Submitting:  20%|██        | 1/5 [00:01<00:05,  1.46s/it]

  [1] Central African mangroves (111):                   single task (n_vertices=2,617)


Submitting:  40%|████      | 2/5 [00:03<00:05,  1.78s/it]

  [2] Myanmar Coast mangroves (321):                   single task (n_vertices=4,783)


Submitting:  60%|██████    | 3/5 [00:07<00:05,  2.88s/it]

  [3] Sunda Shelf mangroves (322):                   single task (n_vertices=9,085)


Submitting:  80%|████████  | 4/5 [00:09<00:02,  2.29s/it]

  [4] Southern Atlantic Brazilian mangroves (616):                   single task (n_vertices=2,490)
  [5] East African mangroves (112):                        year 2003 (n_vertices=71,921)


Submitting:  80%|████████  | 4/5 [00:52<00:13, 13.12s/it]


KeyboardInterrupt: 

In [56]:
# MONITOR TASKS ------------------------------------------------------------------------------------
# NOTE: RUN_ID must match the value used at submission time.
# If the kernel was restarted or it is a different day, manually set:
#   RUN_ID = 'YYYYMMDD'  ← use the date string from _submitted_tasks.txt

print('Monitoring tasks...')
print(f'Filtering for tasks starting with: FIRE_{RUN_ID}_{_run_name}')

while True:
    tasks    = ee.data.getTaskList()
    my_tasks = [t for t in tasks
                if t['description'].startswith(f'FIRE_{RUN_ID}_{_run_name}')]

    ready     = sum(1 for t in my_tasks if t['state'] == 'READY')
    running   = sum(1 for t in my_tasks if t['state'] == 'RUNNING')
    completed = sum(1 for t in my_tasks if t['state'] == 'COMPLETED')
    failed    = sum(1 for t in my_tasks if t['state'] == 'FAILED')

    print(f'  READY: {ready} | RUNNING: {running} | COMPLETED: {completed} | FAILED: {failed}')

    if ready == 0 and running == 0:
        print('All tasks finished.')
        print(f'Failed tasks: {failed}')
        print(f'Move CSVs from Drive/{DRIVE_FOLDER}/ to: {daily_dir}')
        break

    time.sleep(120)

Monitoring tasks...
Filtering for tasks starting with: FIRE_20260413_global_v1
  READY: 6 | RUNNING: 0 | COMPLETED: 17 | FAILED: 0
  READY: 6 | RUNNING: 0 | COMPLETED: 17 | FAILED: 0
  READY: 6 | RUNNING: 0 | COMPLETED: 17 | FAILED: 0


KeyboardInterrupt: 

## Post-Run Assembly

In [ ]:
# FUNCTION: compute_timing_metrics -----------------------------------------------------------------
# Unchanged from the original sequential pipeline.
# Now called locally on downloaded daily count CSVs rather than live GEE data.

def compute_timing_metrics(df, year):
    total = df['n_detections'].sum()
    if total < MIN_DETECTIONS:
        return None

    df     = df.copy().sort_values('doy').reset_index(drop=True)
    doys   = df['doy'].values.astype(float)
    counts = df['n_detections'].values.astype(float)

    cumulative = df['n_detections'].cumsum()
    cum_frac   = cumulative / total

    def doy_at_frac(frac):
        rows = df[cum_frac >= frac]
        return int(rows.iloc[0]['doy']) if not rows.empty else None

    # 1. Primary metrics
    onset_doy     = doy_at_frac(ONSET_THRESHOLD)
    end_doy       = doy_at_frac(END_THRESHOLD)
    rolling       = df['n_detections'].rolling(7, center=True, min_periods=1).mean()
    peak_doy      = int(df.loc[rolling.idxmax(), 'doy'])
    season_length = (end_doy - onset_doy + 1) if (onset_doy and end_doy) else None

    onset_month = ((onset_doy - 1) // 30 + 1) if onset_doy else None
    peak_month  = ((peak_doy  - 1) // 30 + 1) if peak_doy  else None

    peak_outside_window = (
        (onset_doy is not None and end_doy is not None) and
        not (onset_doy <= peak_doy <= end_doy)
    )

    # 2. Alternative thresholds
    onset_doy_10 = doy_at_frac(0.10)
    end_doy_90   = doy_at_frac(0.90)
    onset_doy_15 = doy_at_frac(0.15)
    end_doy_85   = doy_at_frac(0.85)

    # 3. Profile shape
    median_doy      = doy_at_frac(0.50)
    mean_median_div = abs(peak_doy - median_doy) if median_doy is not None else None
    q25_doy         = doy_at_frac(0.25)
    q75_doy         = doy_at_frac(0.75)
    iqr_season_length = (q75_doy - q25_doy + 1) if (q25_doy and q75_doy) else None

    season_mask        = (df['doy'] >= onset_doy) & (df['doy'] <= end_doy)
    active_days        = int((df.loc[season_mask, 'n_detections'] > 0).sum())
    conc_mask          = (df['doy'] >= peak_doy - 45) & (df['doy'] <= peak_doy + 45)
    peak_concentration = round(float(df.loc[conc_mask, 'n_detections'].sum() / total), 4)

    w_mean = (doys * counts).sum() / counts.sum()
    diffs  = doys - w_mean
    w_var  = (counts * diffs**2).sum() / counts.sum()
    w_std  = np.sqrt(w_var)

    if w_std > 0:
        w_skewness     = round(float((counts * (diffs / w_std)**3).sum() / counts.sum()), 4)
        w_kurtosis_raw = round(float((counts * (diffs / w_std)**4).sum() / counts.sum()), 4)
    else:
        w_skewness     = None
        w_kurtosis_raw = None

    # 4. Bimodality
    bc = None
    bimodal_flag_year = 0

    if season_length and season_length > MIN_SEASON_FOR_BIMODALITY:
        n = counts.sum()
        if w_std > 0 and n > 0:
            bc = round(float((w_skewness**2 + 1) / (w_kurtosis_raw + 3 * ((n-1)**2 / ((n-2)*(n-3))))), 4) \
                 if (w_skewness is not None and w_kurtosis_raw is not None) else None

        if bc is not None and bc > BC_THRESHOLD:
            bimodal_flag_year = 1

    return {
        'onset_doy'           : onset_doy,
        'peak_doy'            : peak_doy,
        'end_doy'             : end_doy,
        'season_length'       : season_length,
        'n_detections'        : int(total),
        'onset_month'         : onset_month,
        'peak_month'          : peak_month,
        'peak_outside_window' : int(peak_outside_window),
        'onset_doy_10'        : onset_doy_10,
        'end_doy_90'          : end_doy_90,
        'onset_doy_15'        : onset_doy_15,
        'end_doy_85'          : end_doy_85,
        'median_doy'          : median_doy,
        'mean_median_div'     : mean_median_div,
        'q25_doy'             : q25_doy,
        'q75_doy'             : q75_doy,
        'iqr_season_length'   : iqr_season_length,
        'active_days'         : active_days,
        'peak_concentration'  : peak_concentration,
        'skewness'            : w_skewness,
        'kurtosis_raw'        : w_kurtosis_raw,
        'bc'                  : bc,
        'bimodal_flag_year'   : bimodal_flag_year,
    }

In [ ]:
# POST-RUN ASSEMBLY — DAILY COUNTS + METRICS -------------------------------------------------------
# Reads all per-year CSVs from daily_counts/, concatenates into one daily file,
# then runs compute_timing_metrics() per ecoregion per year to produce metrics CSVs.
#
# Run this after all Drive files have been moved to daily_counts/.

import glob

# --- Assemble daily counts ---
daily_files = sorted(glob.glob(os.path.join(daily_dir, '[!_]*_daily.csv')))
if not daily_files:
    print('No daily CSV files found. Have you moved the Drive exports to daily_counts/?')
else:
    daily_combined = pd.concat(
        [pd.read_csv(f) for f in daily_files], ignore_index=True
    )
    daily_combined.to_csv(os.path.join(output_dir, '_all_daily_counts.csv'), index=False)
    print(f'Daily counts: {len(daily_files)} files → {len(daily_combined)} rows')

    # --- Compute metrics per ecoregion per year ---
    all_metrics  = []
    failed_years = []

    # Build lookup once before the loop
    eco_meta_lookup = {e['eco_id']: e for e in eco_records}

    for eco_id, eco_daily in daily_combined.groupby('eco_id'):
        eco_name   = eco_daily['eco_name'].iloc[0]
        biome_num  = None
        biome_name = None

        # Look up biome info from eco_records
        match = eco_meta_lookup.get(eco_id)
        if match:
            biome_num  = match['biome_num']
            biome_name = match['biome_name']

        eco_metrics = []

        for year, year_group in eco_daily.groupby('year'):
            df_year = year_group[['doy', 'n_detections']].copy()
            metrics = compute_timing_metrics(df_year, year)

            if metrics is not None:
                metrics['eco_id']    = eco_id
                metrics['eco_name']  = eco_name
                metrics['biome_num'] = biome_num
                metrics['biome_name']= biome_name
                metrics['year']      = year
                eco_metrics.append(metrics)
                all_metrics.append(metrics)
            else:
                failed_years.append({
                    'eco_id': eco_id, 'eco_name': eco_name,
                    'year': year, 'reason': 'insufficient_detections'
                })

        n_years_valid   = len(eco_metrics)
        pct_years_valid = round(n_years_valid / len(YEARS), 3)
        for m in eco_metrics:
            m['n_years_valid']   = n_years_valid
            m['pct_years_valid'] = pct_years_valid

        # Save per-ecoregion metrics CSV
        if eco_metrics:
            safe_name = eco_name.replace(' ', '_').replace('/', '_')
            eco_path  = os.path.join(output_dir, f'{eco_id}_{safe_name}.csv')
            pd.DataFrame(eco_metrics).to_csv(eco_path, index=False)

    # Save combined metrics
    if all_metrics:
        metrics_combined = pd.DataFrame(all_metrics)
        metrics_combined.to_csv(os.path.join(output_dir, '_all_metrics.csv'), index=False)
        print(f'Metrics: {len(all_metrics)} ecoregion-year rows across '
              f'{metrics_combined["eco_id"].nunique()} ecoregions')

    # Save failed log
    if failed_years:
        pd.DataFrame(failed_years).to_csv(
            os.path.join(output_dir, '_failed.csv'), index=False
        )
        print(f'Failed years: {len(failed_years)} → _failed.csv')

In [ ]:
# ECOREGION-LEVEL QUALITY METRICS ------------------------------------------------------------------
# Computed across years per ecoregion from the assembled files.
# Items: CV of peak DOY, interannual profile correlation, ecoregion bimodal flag.
#
# NOTE: pct_years_valid (already in _all_metrics.csv) covers fraction of years above
#       detection threshold — no new work needed.
#
# Reads from: _all_metrics.csv and _all_daily_counts.csv
# Writes to:  _eco_quality.csv

metrics_df = pd.read_csv(os.path.join(output_dir, '_all_metrics.csv'))
daily_df   = pd.read_csv(os.path.join(output_dir, '_all_daily_counts.csv'))

print(f'Loaded {len(metrics_df)} ecoregion-year rows across '
      f'{metrics_df["eco_id"].nunique()} ecoregions.')

# -----------------------------------------------------------------------
# CV of peak DOY across years
# Standard deviation / mean of peak_doy across all valid years.
# High CV = peak date is erratic year-to-year.
# -----------------------------------------------------------------------
cv_df = (
    metrics_df.groupby('eco_id')['peak_doy']
    .agg(cv_peak_doy=lambda x: round(float(x.std() / x.mean()), 4)
                                if len(x) > 1 and x.mean() != 0 else None)
    .reset_index()
)

# -----------------------------------------------------------------------
# Ecoregion-level bimodal flag
# Aggregates per-year bimodal_flag_year across all valid years.
# Flag = 1 if > 30% of years were flagged, 0 otherwise.
# -----------------------------------------------------------------------
def eco_bimodal_flag(flags):
    frac_flagged = (flags == 1).sum() / len(flags)
    return 1 if frac_flagged > 0.30 else 0

flag_df = (
    metrics_df.groupby('eco_id')['bimodal_flag_year']
    .agg(
        frac_flagged     = lambda x: round(float((x == 1).sum() / len(x)), 3),
        bimodal_flag_eco = eco_bimodal_flag
    )
    .reset_index()
)

flag_summary = flag_df['bimodal_flag_eco'].value_counts().sort_index()
print(f'\nEcoregion bimodal flag summary:')
print(f'  Clean   (0) : {flag_summary.get(0, 0)}')
print(f'  Flagged (1) : {flag_summary.get(1, 0)}')

# -----------------------------------------------------------------------
# Interannual profile correlation
# For each ecoregion: correlate each year's daily detection profile
# against the long-term mean profile, then average those correlations.
# High mean correlation = consistent season shape year-to-year = reliable metrics.
# Requires >= 3 valid years to compute meaningfully.
# -----------------------------------------------------------------------
profile_corr_rows = []

for eco_id, eco_daily in daily_df.groupby('eco_id'):

    pivot = eco_daily.pivot_table(
        index='year', columns='doy',
        values='n_detections', fill_value=0
    )

    if len(pivot) < 3:
        profile_corr_rows.append({'eco_id': eco_id, 'mean_profile_corr': None})
        continue

    mean_profile = pivot.mean(axis=0).values

    corrs = []
    for yr in pivot.index:
        yr_profile = pivot.loc[yr].values
        if yr_profile.sum() > 0 and mean_profile.sum() > 0:
            r = float(np.corrcoef(yr_profile, mean_profile)[0, 1])
            if not np.isnan(r):
                corrs.append(r)

    mean_corr = round(float(np.mean(corrs)), 4) if corrs else None
    profile_corr_rows.append({'eco_id': eco_id, 'mean_profile_corr': mean_corr})

corr_df = pd.DataFrame(profile_corr_rows)

# -----------------------------------------------------------------------
# MERGE AND SAVE
# -----------------------------------------------------------------------
eco_quality = (
    cv_df
    .merge(flag_df,  on='eco_id')
    .merge(corr_df,  on='eco_id')
)

eco_quality_path = os.path.join(output_dir, '_eco_quality.csv')
eco_quality.to_csv(eco_quality_path, index=False)

print(f'\nEcoregion quality metrics saved: {len(eco_quality)} ecoregions')
print(f'Path: {os.path.abspath(eco_quality_path)}')
print()
print(eco_quality.head(10).to_string())

In [ ]:
# COMBINE ALL RESULTS INTO MASTER CSV --------------------------------------------------------------
# Reads from _all_metrics.csv and _eco_quality.csv (both on disk).
# Ecoregion-level quality columns are broadcast to every row for that ecoregion.

master_df   = pd.read_csv(os.path.join(output_dir, '_all_metrics.csv'))
eco_quality = pd.read_csv(os.path.join(output_dir, '_eco_quality.csv'))

master_df = master_df.merge(eco_quality, on='eco_id', how='left')

col_order = [
    'eco_id', 'eco_name', 'biome_num', 'biome_name', 'year',
    # Primary metrics
    'onset_doy', 'peak_doy', 'end_doy', 'season_length',
    'n_detections', 'onset_month', 'peak_month',
    # Alternative thresholds
    'onset_doy_10', 'end_doy_90',
    'onset_doy_15', 'end_doy_85',
    # Profile shape
    'median_doy', 'mean_median_div', 'q25_doy', 'q75_doy',
    'iqr_season_length', 'active_days',
    'peak_concentration', 'skewness', 'kurtosis_raw',
    # Bimodality diagnostics (per year)
    'bc', 'bimodal_flag_year',
    # Per-year quality
    'peak_outside_window', 'n_years_valid', 'pct_years_valid',
    # Ecoregion-level quality
    'cv_peak_doy', 'mean_profile_corr',
    'frac_flagged', 'bimodal_flag_eco',
]

col_order = [c for c in col_order if c in master_df.columns]
master_df = master_df[col_order]

master_path = os.path.join(output_dir, f'master_{_run_name}.csv')
master_df.to_csv(master_path, index=False)

print(f'Master CSV: {master_df.shape[0]} rows × {master_df.shape[1]} columns')
print(f'Path: {os.path.abspath(master_path)}')
print()
print(master_df.head(10).to_string())